In [35]:
import pandas as pd
import requests
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

In [ ]:
data_actornames = pd.read_csv("../data/name.basics.tsv", sep="\t")
data_actornames = data_actornames.drop(columns=["deathYear", "birthYear"])
data_actornames = data_actornames[data_actornames["primaryProfession"].str.contains("actor|actress", na=False)]
data_actornames = data_actornames[data_actornames["knownForTitles"].str.contains(",")]
#data_actornames.to_csv("../data/actor_names.csv", index=False)
#data_actornames.to_parquet("../data/name.basics.parquet", engine="pyarrow", index=False)

In [ ]:
def invert_mapping(
    df: pd.DataFrame,
    id_col: str = "nconst",
    titles_col: str = "knownForTitles",
    null_value: str = "\\N",
) -> pd.DataFrame:
    result = df[[id_col, titles_col]].dropna()
    result = result[result[titles_col] != null_value]

    result = result.copy()
    result[titles_col] = result[titles_col].str.split(",")
    result = result.explode(titles_col)
    result[titles_col] = result[titles_col].str.strip()
    result = result[result[titles_col] != ""]

    movie_df = (
        result.groupby(titles_col)[id_col]
        .apply(lambda x: ",".join(sorted(x.unique())))
        .reset_index()
    )
    movie_df.columns = ["tconst", "personIds"]

    return movie_df.sort_values("tconst").reset_index(drop=True)


In [5]:
movie_list = invert_mapping(data_actornames)
movie_list = movie_list[movie_list["personIds"].str.contains(",")]

In [6]:
title_basics = pd.read_csv("../data/title.basics.tsv", sep="\t")
title_basics = title_basics[title_basics["titleType"] == "movie"]
title_basics["startYear"] = pd.to_numeric(title_basics["startYear"], errors="coerce")
title_basics = title_basics[(title_basics["startYear"] > 1957) & (title_basics["startYear"] <= 2026)]
title_basics = title_basics[~title_basics["genres"].str.contains("Documentary", na=False)]
title_basics = title_basics.drop(columns=["titleType", "isAdult", "endYear", "runtimeMinutes", "genres"])

In [21]:
merge_movielist = movie_list.merge(title_basics, on="tconst", how="inner")
#merge_movielist.to_csv("../data/movie_data.csv", index=False)
#merge_movielist.to_parquet("../data/movie_data.parquet", engine="pyarrow", index=False)

In [21]:
movie_list = pd.read_parquet("../data/movie_data.parquet", engine="pyarrow")

In [22]:
movie_list

,tconst,personIds,primaryTitle,originalTitle,startYear
0,tt0015724,"nm0190923,nm0328456,nm0600039,nm0650495,nm0670...",Dama de noche,Dama de noche,1993.0
1,tt0035423,"nm0043662,nm0118806,nm0126148,nm0179296,nm0294...",Kate & Leopold,Kate & Leopold,2001.0
2,tt0036606,"nm0028791,nm0336942,nm0426834,nm0485392,nm0517...","Another Time, Another Place","Another Time, Another Place",1983.0
3,tt0038086,"nm0076617,nm0294590",Shiva und die Galgenblume,Shiva und die Galgenblume,1993.0
4,tt0039442,"nm0206873,nm0210383,nm0349426,nm0481173,nm0631...","Habla, mudita","Habla, mudita",1973.0
...,...,...,...,...,...
303942,tt9916270,"nm3645214,nm3701625,nm4703427,nm6366028,nm6411...",Il talento del calabrone,Il talento del calabrone,2020.0
303943,tt9916362,"nm0357571,nm10678594,nm10678595,nm10678597,nm1...",Coven,Akelarre,2020.0
303944,tt9916428,"nm1316038,nm15908324,nm3286302,nm3710453,nm859...",The Secret of China,Hong xing zhao yao Zhong guo,2019.0
303945,tt9916538,"nm10041459,nm10538477,nm10538478,nm10538479,nm...",Kuambil Lagi Hatiku,Kuambil Lagi Hatiku,2019.0


In [15]:
TMDB_API_KEY = "cc0093b5ad09a876190e097a1c2c8e65"

def is_english_movie(imdb_id: str) -> dict:
    tmdb_url = f"https://api.themoviedb.org/3/find/{imdb_id}"
    params = {
        "api_key": TMDB_API_KEY,
        "external_source": "imdb_id"
    }

    try:
        response = requests.get(tmdb_url, params=params)
        response.raise_for_status()
        data = response.json()

        movies = data.get("movie_results", [])
        if not movies:
            return {"tconst": imdb_id, "original_language": None}
        
        movie = movies[0]
        original_language = movie.get("original_language")

        return {
            "tconst": imdb_id,
            "original_language": original_language
        }
    
    except requests.exceptions.RequestException as e:
        return {"tconst": imdb_id, "original_language": None}

In [ ]:
def check_movies(df: pd.DataFrame, imdb_id_col: str = "imdb_id", max_workers: int = 20, checkpoint_file: str = "checkpoint.csv") -> pd.DataFrame:
    imdb_ids = df[imdb_id_col].tolist()
    results = [None] * len(imdb_ids)
    completed = 0
    last_checkpoint = 0

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(is_english_movie, imdb_id): i for i, imdb_id in enumerate(imdb_ids)}

        for future in as_completed(futures):
            idx = futures[future]
            completed += 1
            results[idx] = future.result()

            if completed % 100 == 0:
                print(f"Progress: {completed}/{len(imdb_ids)}")

            if completed - last_checkpoint >= 25000:
                checkpoint_df = pd.DataFrame([r for r in results if r is not None])
                checkpoint_df.to_csv(checkpoint_file, index=False)
                print(f"Checkpoint saved at {completed} records -> {checkpoint_file}")
                last_checkpoint = completed

    # Final save for any remaining records
    results_df = pd.DataFrame(results)
    results_df.to_csv(checkpoint_file, index=False)
    print(f"Final checkpoint saved -> {checkpoint_file}")

    return df.merge(results_df, left_on=imdb_id_col, right_on="imdb_id", how="left")